# ESM2+GRU Test Predictions for Competition

This notebook generates predictions using the trained **ESM-2 + GRU** model.

## Pipeline:
1. Load test data (59,540 sequences)
2. Load ESM-2 model for embedding extraction
3. Extract embeddings from test sequences
4. Load trained GRU model
5. Generate predictions
6. Apply optimal thresholds (or 0.5 default)
7. Save submission.csv

## 1. Configuration

In [15]:
import os

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input file
TEST_INPUT = "../data/independent_test_input.csv"

# Trained model paths
MODEL_DIR = "../output/transformer_gru/run02_frozen_esm2_augmented_v2"
MODEL_FILE = f"{MODEL_DIR}/transformer_gru_model_best.h5"
OPTIMAL_THRESHOLDS_FILE = f"{MODEL_DIR}/optimal_thresholds.pkl"

# ESM-2 model
ESM_MODEL_NAME = "esm2_t12_35M_UR50D"  # Same as training

# Output
OUTPUT_DIR = "../output/test_predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Batch size for processing
BATCH_SIZE = 32

# Use optimal thresholds or default 0.5?
USE_OPTIMAL_THRESHOLDS = True  # Set to False to use 0.5

print("Configuration loaded:")
print(f"  Test input: {TEST_INPUT}")
print(f"  Model: {MODEL_FILE}")
print(f"  ESM-2: {ESM_MODEL_NAME}")
print(f"  Use optimal thresholds: {USE_OPTIMAL_THRESHOLDS}")
print(f"  Output: {OUTPUT_DIR}/submission.csv")

Configuration loaded:
  Test input: ../data/independent_test_input.csv
  Model: ../output/transformer_gru/run02_frozen_esm2_augmented_v2/transformer_gru_model_best.h5
  ESM-2: esm2_t12_35M_UR50D
  Use optimal thresholds: True
  Output: ../output/test_predictions/submission.csv


## 2. Import Libraries

In [16]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras

# ESM-2
import torch
import esm

print(f"✓ Libraries imported")
print(f"  TensorFlow: {tf.__version__}")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

✓ Libraries imported
  TensorFlow: 2.20.0
  PyTorch: 2.4.1+cu121
  Device: cpu


## 3. Load Test Data

In [17]:
# Load test data
test_df = pd.read_csv(TEST_INPUT)

print(f"✓ Loaded test data")
print(f"  Samples: {len(test_df):,}")
print(f"  Columns: {list(test_df.columns)}")
print(f"\nFirst few rows:")
test_df.head()

✓ Loaded test data
  Samples: 59,540
  Columns: ['ID', 'Sequence']

First few rows:


,ID,Sequence
0,Q0GA42,AAAAAAAALGVRLRDCCSRGAVLLLFFSLSP
1,Q0GA42,AAAAAAALGVRLRDCCSRGAVLLLFFSLSPR
2,P11047,AAAAAAGCAQAAMDECTDEGGRPQRCMPEFV
3,Q7TNS5,AAAAAASSASSPATRCKELGLAAAAAWEQQG
4,Q9NV92,AAAAAETSQRIQEEECPPRDDFSDADQLRVG


## 4. Load ESM-2 Model

In [18]:
print(f"Loading ESM-2 model: {ESM_MODEL_NAME}...")

esm_model, alphabet = esm.pretrained.load_model_and_alphabet(ESM_MODEL_NAME)
esm_model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
esm_model = esm_model.to(device)

print(f"✓ ESM-2 model loaded")
print(f"  Device: {device}")
print(f"  Embedding dim: {esm_model.embed_dim}")
print(f"  Layers: {esm_model.num_layers}")

Loading ESM-2 model: esm2_t12_35M_UR50D...
✓ ESM-2 model loaded
  Device: cpu
  Embedding dim: 480
  Layers: 12


## 5. Extract Embeddings from Test Sequences

In [19]:
def extract_esm2_embeddings(sequences, model, alphabet, batch_size=32, device='cpu'):
    """
    Extract per-residue embeddings from ESM-2 model.
    """
    model = model.to(device)
    model.eval()
    
    batch_converter = alphabet.get_batch_converter()
    all_embeddings = []
    
    print(f"Extracting embeddings for {len(sequences):,} sequences...")
    
    for i in range(0, len(sequences), batch_size):
        batch_sequences = sequences[i:i+batch_size]
        
        # Prepare batch
        batch_labels = [(f"seq_{j}", seq) for j, seq in enumerate(batch_sequences)]
        _, _, batch_tokens = batch_converter(batch_labels)
        batch_tokens = batch_tokens.to(device)
        
        # Extract embeddings
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[model.num_layers])
            embeddings = results["representations"][model.num_layers]
            
            # Remove special tokens
            embeddings = embeddings[:, 1:-1, :]
            
            all_embeddings.append(embeddings.cpu().numpy())
        
        if (i + batch_size) % 1000 == 0 or (i + batch_size) >= len(sequences):
            print(f"  Processed {min(i+batch_size, len(sequences)):,} / {len(sequences):,} sequences")
    
    # Concatenate all batches
    all_embeddings = np.concatenate(all_embeddings, axis=0)
    print(f"\n✓ Embeddings extracted: {all_embeddings.shape}")
    
    return all_embeddings

# Extract embeddings
test_sequences = test_df['Sequence'].values
X_test_embed = extract_esm2_embeddings(
    test_sequences,
    esm_model,
    alphabet,
    batch_size=BATCH_SIZE,
    device=device
)

Extracting embeddings for 59,540 sequences...
  Processed 4,000 / 59,540 sequences
  Processed 8,000 / 59,540 sequences
  Processed 12,000 / 59,540 sequences
  Processed 16,000 / 59,540 sequences
  Processed 20,000 / 59,540 sequences
  Processed 24,000 / 59,540 sequences
  Processed 28,000 / 59,540 sequences
  Processed 32,000 / 59,540 sequences
  Processed 36,000 / 59,540 sequences
  Processed 40,000 / 59,540 sequences
  Processed 44,000 / 59,540 sequences
  Processed 48,000 / 59,540 sequences
  Processed 52,000 / 59,540 sequences
  Processed 56,000 / 59,540 sequences
  Processed 59,540 / 59,540 sequences

✓ Embeddings extracted: (59540, 31, 480)


## 6. Load Trained GRU Model

In [20]:
print(f"Loading trained model from {MODEL_FILE}...")

# Load model
model = keras.models.load_model(MODEL_FILE, compile=False)

print(f"✓ Model loaded")
print(f"\nModel summary:")
model.summary()

Loading trained model from ../output/transformer_gru/run02_frozen_esm2_augmented_v2/transformer_gru_model_best.h5...
✓ Model loaded

Model summary:


Model: "transformer_gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_input (InputLayer)    │ (None, 31, 480)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_gru_1             │ (None, 31, 256)        │       468,480 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_gru_2             │ (None, 128)            │       123,648 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_1 (Activation)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_dense_1 (Dropout)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_2 (Activation)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_dense_2 (Dropout)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 617,859 (2.36 MB)

 Trainable params: 617,475 (2.36 MB)

 Non-trainable params: 384 (1.50 KB)

## 7. Generate Predictions

In [21]:
print("Generating predictions...")

# Get probability predictions
test_pred_proba = model.predict(X_test_embed, batch_size=BATCH_SIZE, verbose=1)

print(f"\n✓ Predictions generated")
print(f"  Shape: {test_pred_proba.shape}")
print(f"  Probability range: [{test_pred_proba.min():.4f}, {test_pred_proba.max():.4f}]")

Generating predictions...
1861/1861 ━━━━━━━━━━━━━━━━━━━━ 59s 31ms/step

✓ Predictions generated
  Shape: (59540, 3)
  Probability range: [0.0000, 0.9982]


## 8. Apply Thresholds

In [22]:
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]

# Load optimal thresholds if available
if USE_OPTIMAL_THRESHOLDS and os.path.exists(OPTIMAL_THRESHOLDS_FILE):
    with open(OPTIMAL_THRESHOLDS_FILE, 'rb') as f:
        optimal_thresholds = pickle.load(f)
    print("Using optimal thresholds:")
    for label, threshold in optimal_thresholds.items():
        print(f"  {label}: {threshold:.3f}")
    
    # Apply per-label thresholds
    binary_pred = np.zeros_like(test_pred_proba, dtype=int)
    for i, label in enumerate(label_cols):
        threshold = optimal_thresholds[label]
        binary_pred[:, i] = (test_pred_proba[:, i] > threshold).astype(int)
else:
    print("Using default threshold: 0.5")
    binary_pred = (test_pred_proba > 0.5).astype(int)

print(f"\n✓ Binary predictions generated")
print(f"\nPredicted positive rates:")
for i, label in enumerate(label_cols):
    pos_count = binary_pred[:, i].sum()
    pos_rate = pos_count / len(binary_pred) * 100
    print(f"  {label}: {pos_count:,} ({pos_rate:.2f}%)")

Using optimal thresholds:
  S-glutathionylation: 0.180
  S-nitrosylation: 0.270
  S-palmitoylation: 0.300

✓ Binary predictions generated

Predicted positive rates:
  S-glutathionylation: 5,326 (8.95%)
  S-nitrosylation: 19,787 (33.23%)
  S-palmitoylation: 1,646 (2.76%)


## 9. Create Submission File

In [23]:
# Create submission DataFrame
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Sequence': test_df['Sequence'],
    'S-glutathionylation': binary_pred[:, 0],
    'S-nitrosylation': binary_pred[:, 1],
    'S-palmitoylation': binary_pred[:, 2]
})

# Save to CSV
submission_file = f"{OUTPUT_DIR}/submission_esm2_gru.csv"
submission.to_csv(submission_file, index=False)

print("\n" + "="*80)
print("✓ SUBMISSION FILE CREATED!")
print("="*80)
print(f"\nFile: {submission_file}")
print(f"Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
display(submission.head(10))

print(f"\n{'='*80}")
print("✓ Ready for submission!")
print(f"{'='*80}")


✓ SUBMISSION FILE CREATED!

File: ../output/test_predictions/submission_esm2_gru.csv
Shape: (59540, 5)

First 10 rows:


,ID,Sequence,S-glutathionylation,S-nitrosylation,S-palmitoylation
0,Q0GA42,AAAAAAAALGVRLRDCCSRGAVLLLFFSLSP,0,0,0
1,Q0GA42,AAAAAAALGVRLRDCCSRGAVLLLFFSLSPR,0,0,0
2,P11047,AAAAAAGCAQAAMDECTDEGGRPQRCMPEFV,1,1,0
3,Q7TNS5,AAAAAASSASSPATRCKELGLAAAAAWEQQG,0,1,0
4,Q9NV92,AAAAAETSQRIQEEECPPRDDFSDADQLRVG,0,1,0
5,Q6ZN55,AAAAAQAPRRFECGTCGKKVGSAARLQAHEA,0,0,0
6,P52746,AAAAEPLPLRCFQEGCSYAAPDRKAFIKHLK,0,0,0
7,Q9HB90,AAAAGGGVGAGAGGGCGPGGADSSKPRILLM,0,1,0
8,Q96FX8,AAAAMLFCGFIILVICFILSFFALCGPQMLV,0,0,0
9,Q9VP57,AAAANENLVGTSLYTCNHCQFKSTDKVVFDE,0,0,0



✓ Ready for submission!


## 10. Optional: Save Probabilities

In [24]:
# Save probabilities for threshold tuning later
prob_file = f"{OUTPUT_DIR}/test_probabilities_esm2_gru.csv"

prob_df = pd.DataFrame({
    'ID': test_df['ID'],
    'Sequencce' : test_df['Sequence']
    'glut_proba': test_pred_proba[:, 0],
    'nitro_proba': test_pred_proba[:, 1],
    'palm_proba': test_pred_proba[:, 2]
})

prob_df.to_csv(prob_file, index=False)
print(f"✓ Saved probabilities to {prob_file}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2893453937.py, line 6)